In [ ]:
import numpy as np

def logistic_function(x):
    """
    Computes the logistic (sigmoid) function applied to any value of x.
    Arguments:
        x: scalar or numpy array of any size.
    Returns:
        y: logistic function applied to x.
    """
    return 1 / (1 + np.exp(-x))

def test_logistic_function():
    x_scalar = 0
    assert round(logistic_function(x_scalar), 3) == 0.5
    
    x_pos = 2
    assert round(logistic_function(x_pos), 3) == 0.881
    
    x_neg = -3
    assert round(logistic_function(x_neg), 3) == 0.047
    
    x_array = np.array([0, 2, -3])
    expected_output_array = np.array([0.5, 0.881, 0.047])
    assert np.all(np.round(logistic_function(x_array), 3) == expected_output_array)
    
    print("All tests passed!")

test_logistic_function()


All tests passed!


In [ ]:
import numpy as np

def log_loss(y_true, y_pred):
    """
    Computes log loss for true target value y {0 or 1} and predicted target value y' in [0,1].
    """
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    return - (y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def test_log_loss():
    assert np.isclose(log_loss(1, 1), 0.0)
    assert np.isclose(log_loss(0, 0), 0.0)
    assert np.isclose(log_loss(1, 0.8), -np.log(0.8), atol=1e-6)
    assert np.isclose(log_loss(0, 0.2), -np.log(0.8), atol=1e-6)
    print("All tests passed!")

test_log_loss()


All tests passed!


In [ ]:
import numpy as np

def cost_function(y_true, y_pred):
    """
    Computes average log loss for arrays of true values and predicted probabilities.
    Args:
        y_true (array_like): true target values (0 or 1), shape (n,)
        y_pred (array_like): predicted probabilities, shape (n,)
    Returns:
        cost (float): average log loss
    """
    assert len(y_true) == len(y_pred), "Length mismatch between y_true and y_pred"
    n = len(y_true)
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    loss_vec = - (y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    cost = np.mean(loss_vec)
    return cost

def test_cost_function():
    y_true = np.array([1, 0, 1])
    y_pred = np.array([0.9, 0.1, 0.8])
    expected_cost = (-(np.log(0.9)) + -(np.log(0.9)) + -(np.log(0.8))) / 3
    result = cost_function(y_true, y_pred)
    assert np.isclose(result, expected_cost, atol=1e-6), f"Test failed: {result} != {expected_cost}"
    print("Test passed for cost_function!")

test_cost_function()


Test passed for cost_function!


In [ ]:
def costfunction_logreg(X, y, w, b):
    """
    Computes the logistic regression cost function using vectorization.
    
    Args:
        X (ndarray, shape (n,d)): feature matrix
        y (array_like, shape (n,)): true labels {0,1}
        w (array_like, shape (d,)): weight parameters
        b (float): bias parameter
        
    Returns:
        cost (float): average log loss
    """
    n, d = X.shape
    assert len(y) == n, "Number of samples in X and y must match"
    assert len(w) == d, "Number of features and weights must match"
    
    z = np.dot(X, w) + b
    y_pred = logistic_function(z)
    cost = cost_function(y, y_pred)
    return cost

X = np.array([[10, 20], [-10, 10]])
y = np.array([1, 0])
w = np.array([0.5, 1.5])
b = 1
print("Cost for logistic regression:", costfunction_logreg(X, y, w, b))


Cost for logistic regression: 5.500008350834906


In [ ]:
def compute_gradient(X, y, w, b):
    """
    Computes gradients of the cost function with respect to model parameters.
    Args:
        X (ndarray, shape (n,d)): Input data
        y (array_like, shape (n,)): True labels (0 or 1)
        w (array_like, shape (d,)): Weight parameters
        b (float): Bias parameter
    Returns:
        grad_w (array_like, shape (d,)): Gradient w.r.t weights
        grad_b (float): Gradient w.r.t bias
    """
    n, d = X.shape
    assert len(y) == n, f"Expected y to have {n} elements, got {len(y)}"
    assert len(w) == d, f"Expected w to have {d} elements, got {len(w)}"
    
    z = np.dot(X, w) + b
    y_pred = logistic_function(z)

    grad_w = np.dot(X.T, (y_pred - y)) / n
    grad_b = np.sum(y_pred - y) / n
    
    return grad_w, grad_b

X = np.array([[10, 20], [-10, 10]])  
y = np.array([1, 0])           
w = np.array([0.5, 1.5])  
b = 1                  

grad_w, grad_b = compute_gradient(X, y, w, b)
print("Gradient w:", grad_w)
print("Gradient b:", grad_b)


Gradient w: [-4.99991649  4.99991649]
Gradient b: 0.4999916492890759


In [ ]:
def gradient_descent(X, y, w, b, alpha, n_iter, show_cost=False, show_params=False):
    """
    Implements batch gradient descent to optimize logistic regression parameters.
    """
    n, d = X.shape
    assert len(y) == n, "Number of observations in X and y do not match"
    assert len(w) == d, "Number of features in X and w do not match"
    
    cost_history = []
    params_history = []
    
    for i in range(n_iter):
        grad_w, grad_b = compute_gradient(X, y, w, b)
        
        w -= alpha * grad_w
        b -= alpha * grad_b

        cost = costfunction_logreg(X, y, w, b)
        
        cost_history.append(cost)
        params_history.append((w.copy(), b))

        if show_cost and (i % 100 == 0 or i == n_iter - 1):
            print(f"Iteration {i}: Cost = {cost:.6f}")
        if show_params and (i % 100 == 0 or i == n_iter - 1):
            print(f"Iteration {i}: w = {w}, b = {b:.6f}")
    
    return w, b, cost_history, params_history

X = np.array([[0.1, 0.2], [-0.1, 0.1]]) 
y = np.array([1, 0])                 
w = np.zeros(X.shape[1])         
b = 0.0                            
alpha = 0.1                              
n_iter = 500                      
w_out, b_out, cost_history, params_history = gradient_descent(
    X, y, w, b, alpha, n_iter, show_cost=True, show_params=False
)

print("\nFinal parameters:")
print("w:", w_out)
print("b:", b_out)
print("Final cost:", cost_history[-1])


Iteration 0: Cost = 0.692835
Iteration 100: Cost = 0.662662
Iteration 200: Cost = 0.634332
Iteration 300: Cost = 0.607704
Iteration 400: Cost = 0.582671
Iteration 499: Cost = 0.559357

Final parameters:
w: [2.31611022 1.13447251]
b: -0.15721731010662016
Final cost: 0.5593567092003469


In [ ]:
import numpy as np

def prediction(X, w, b, threshold=0.5):
    """
    Predicts binary outcomes for given input features based on logistic regression parameters.
    """
    z = np.dot(X, w) + b
    y_prob = logistic_function(z)
    
    y_pred = (y_prob >= threshold).astype(int)
    return y_pred

def test_prediction():
    X_test = np.array([[0.5, 1.0], [1.5, -0.5], [-0.5, -1.0]])
    w_test = np.array([1.0, -1.0]) 
    b_test = 0.0
    threshold = 0.5
    
    expected_output = np.array([0, 1, 1]) 
    y_pred = prediction(X_test, w_test, b_test, threshold)
    
    assert np.array_equal(y_pred, expected_output), f"Expected {expected_output}, got {y_pred}"
    print("Test passed for prediction function!")

test_prediction()


Test passed for prediction function!


In [ ]:
def evaluate_classification(y_true, y_pred):
    """
    Computes confusion matrix, precision, recall, and F1-score for binary classification.
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    confusion_matrix = np.array([[TN, FP],
                                 [FN, TP]])
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    
    metrics = {
        "confusion_matrix": confusion_matrix,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score
    }
    return metrics

y_true = np.array([1, 0, 1, 0, 1])
y_pred = np.array([1, 0, 0, 0, 1])

metrics = evaluate_classification(y_true, y_pred)
print("Confusion Matrix:\n", metrics["confusion_matrix"])
print("Precision:", metrics["precision"])
print("Recall:", metrics["recall"])
print("F1-score:", metrics["f1_score"])


Confusion Matrix:
 [[2 0]
 [1 2]]
Precision: 1.0
Recall: 0.6666666666666666
F1-score: 0.8


part 2


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = pd.read_csv("Iris.csv")

iris['BinaryLabel'] = (iris['Species'] == 'Iris-setosa').astype(int)

X = iris[['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']].values
y = iris['BinaryLabel'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [ ]:
w = np.zeros(X_train_scaled.shape[1])
b = 0.0
alpha = 0.1
n_iter = 1000

w_out, b_out, cost_history, params_history = gradient_descent(
    X_train_scaled, y_train, w, b, alpha, n_iter, show_cost=True, show_params=False
)

print("\nFinal parameters:")
print("w:", w_out)
print("b:", b_out)
print("Final cost:", cost_history[-1])


Iteration 0: Cost = 0.637317
Iteration 100: Cost = 0.077597
Iteration 200: Cost = 0.043467
Iteration 300: Cost = 0.030551
Iteration 400: Cost = 0.023695
Iteration 500: Cost = 0.019422
Iteration 600: Cost = 0.016494
Iteration 700: Cost = 0.014357
Iteration 800: Cost = 0.012727
Iteration 900: Cost = 0.011440
Iteration 999: Cost = 0.010407

Final parameters:
w: [-0.9360863   1.95173718 -2.00897128 -1.86907666]
b: -2.0057119707934423
Final cost: 0.010406742591730305


In [ ]:
y_pred_test = prediction(X_test_scaled, w_out, b_out)

metrics = evaluate_classification(y_test, y_pred_test)
print("\nConfusion Matrix:\n", metrics["confusion_matrix"])
print("Precision:", metrics["precision"])
print("Recall:", metrics["recall"])
print("F1-score:", metrics["f1_score"])



Confusion Matrix:
 [[20  0]
 [ 0 10]]
Precision: 1.0
Recall: 1.0
F1-score: 1.0


In [ ]:
import numpy as np

def softmax(z):
    """
    Compute softmax probabilities for each row of z.
    z: numpy array of shape (n, c)
    Returns: numpy array of same shape with probabilities
    """
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

z = np.array([[1,2,3]])
print("Softmax:", softmax(z))


Softmax: [[0.09003057 0.24472847 0.66524096]]


In [14]:
def loss_softmax(y_true, y_pred):
    """
    Cross-entropy loss for one-hot encoded labels.
    """
    return -np.sum(y_true * np.log(y_pred + 1e-10))


In [15]:
def cost_softmax(X, y, W, b):
    """
    Compute average cross-entropy cost.
    X: (n,d), y: (n,c) one-hot, W: (d,c), b: (c,)
    """
    n, d = X.shape
    z = np.dot(X, W) + b
    y_pred = softmax(z)
    return -np.sum(y * np.log(y_pred + 1e-10)) / n


In [16]:
def compute_gradient_softmax(X, y, W, b):
    """
    Compute gradients for softmax regression.
    """
    n, d = X.shape
    z = np.dot(X, W) + b
    y_pred = softmax(z)
    grad_W = np.dot(X.T, (y_pred - y)) / n
    grad_b = np.sum(y_pred - y, axis=0) / n
    return grad_W, grad_b


In [17]:
def gradient_descent_softmax(X, y, W, b, alpha, n_iter, show_cost=False):
    cost_history = []
    for i in range(n_iter):
        grad_W, grad_b = compute_gradient_softmax(X, y, W, b)
        W -= alpha * grad_W
        b -= alpha * grad_b
        cost = cost_softmax(X, y, W, b)
        cost_history.append(cost)
        if show_cost and (i % 100 == 0 or i == n_iter-1):
            print(f"Iteration {i}: Cost = {cost:.6f}")
    return W, b, cost_history


In [18]:
def predict_softmax(X, W, b):
    """
    Predict class labels using softmax regression.
    """
    z = np.dot(X, W) + b
    y_pred = softmax(z)
    return np.argmax(y_pred, axis=1)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = iris[['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']].values
y = iris['Species'].values.reshape(-1,1)

encoder = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

n, d = X_train_scaled.shape
c = y_train.shape[1]
W = np.zeros((d,c))
b = np.zeros(c)

W_out, b_out, cost_history = gradient_descent_softmax(X_train_scaled, y_train, W, b, alpha=0.1, n_iter=1000, show_cost=True)

y_pred_test = predict_softmax(X_test_scaled, W_out, b_out)
y_true_test = np.argmax(y_test, axis=1)

confusion_matrix = np.zeros((c,c), dtype=int)
for i in range(len(y_true_test)):
    confusion_matrix[y_true_test[i], y_pred_test[i]] += 1

accuracy = np.mean(y_true_test == y_pred_test)

print("\nConfusion Matrix:\n", confusion_matrix)
print("Accuracy:", accuracy)


Iteration 0: Cost = 1.007146
Iteration 100: Cost = 0.320108
Iteration 200: Cost = 0.251776
Iteration 300: Cost = 0.211968
Iteration 400: Cost = 0.185130
Iteration 500: Cost = 0.165809
Iteration 600: Cost = 0.151254
Iteration 700: Cost = 0.139902
Iteration 800: Cost = 0.130800
Iteration 900: Cost = 0.123337
Iteration 999: Cost = 0.117161

Confusion Matrix:
 [[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
Accuracy: 0.9333333333333333
